In [1]:
import socket

In [2]:
import threading

In [3]:
import time

In [4]:
HOST = "127.0.0.1"

In [5]:
PORT = 65002

In [6]:
MAX_WORKERS = 2

# A Semaphore is like a Lock, but it hands out N permits instead of 1.
# Up to MAX_WORKERS threads can hold a permit at once; anyone else
# calling .acquire() (via the "with" statement) blocks until a permit
# is released.

In [7]:
worker_slots = threading.Semaphore(MAX_WORKERS)

In [8]:
def handle_client(conn, addr):
    with conn:
        request = conn.recv(1024).decode()

        print(f"[WORKER-{threading.get_ident()}] Waiting for a free slot ({addr})")

        # CHANGED: acquire() now takes a timeout instead of blocking forever
        got_slot = worker_slots.acquire(timeout=3)

        if not got_slot:
            # CHANGED: if we waited 3s and still got no slot, reject the client
            print(f"[WORKER-{threading.get_ident()}] No slot free, rejecting {addr}")
            conn.sendall(b"SERVER BUSY: please try again later")
            return

        try:
            print(f"[WORKER-{threading.get_ident()}] Got a slot, working on {addr}")
            time.sleep(2)
            reply = f"Processed '{request}' by worker thread {threading.get_ident()}"
            conn.sendall(reply.encode())
            print(f"[WORKER-{threading.get_ident()}] Done with {addr}, releasing slot")
        finally:
            worker_slots.release()  # CHANGED: must release manually now, no more "with"

In [9]:
def main():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as server_socket:
        server_socket.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        server_socket.bind((HOST, PORT))
        server_socket.listen(5)
        print(f"[DISPATCHER] Listening on {HOST}:{PORT} (max {MAX_WORKERS} concurrent workers)")

        while True:
            conn, addr = server_socket.accept()  # still accepts immediately, no change here
            print(f"[DISPATCHER] Accepted {addr}, spawning worker thread")
            worker = threading.Thread(target=handle_client, args=(conn, addr))
            worker.start()  # thread starts, but may block INSIDE on the semaphore

In [ ]:
if __name__ == "__main__":
    main()

[DISPATCHER] Listening on 127.0.0.1:65002 (max 2 concurrent workers)
[DISPATCHER] Accepted ('127.0.0.1', 51186), spawning worker thread
[WORKER-9064] Waiting for a free slot (('127.0.0.1', 51186))
[WORKER-9064] Got a slot, working on ('127.0.0.1', 51186)
[DISPATCHER] Accepted ('127.0.0.1', 51187), spawning worker thread
[DISPATCHER] Accepted ('127.0.0.1', 51188), spawning worker thread
[WORKER-6716] Waiting for a free slot (('127.0.0.1', 51187))
[WORKER-6716] Got a slot, working on ('127.0.0.1', 51187)
[WORKER-17816] Waiting for a free slot (('127.0.0.1', 51188))
[DISPATCHER] Accepted ('127.0.0.1', 51189), spawning worker thread
[WORKER-7776] Waiting for a free slot (('127.0.0.1', 51189))
[DISPATCHER] Accepted ('127.0.0.1', 51190), spawning worker thread
[WORKER-8364] Waiting for a free slot (('127.0.0.1', 51190))
[WORKER-9064] Done with ('127.0.0.1', 51186), releasing slot
[WORKER-17816] Got a slot, working on ('127.0.0.1', 51188)
[WORKER-6716] Done with ('127.0.0.1', 51187), releasin